<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
LCEL Chains
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
# Wichtig: LangSmith-Account und API-Key im EU-Workspace anlegen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M05-LCEL-Chains"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

In diesem Modul bauen Sie LCEL-Chains von einfach bis modular.

**Lernziele:**
- LCEL-Grundidee verstehen (`Runnable`-Bausteine)
- mit dem Pipe-Operator `|` Chains bauen
- mehrere Verarbeitungsschritte kombinieren
- `invoke()` und `stream()` sinnvoll einsetzen


In [ ]:
# Imports für LCEL-Beispiele
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
parser = StrOutputParser()
print("LCEL Setup bereit")

# 2 | Was ist LCEL?
---

LCEL (LangChain Expression Language) beschreibt Pipelines aus `Runnable`-Bausteinen.
Jeder Schritt hat klaren Input und Output.

Typischer Flow:
`Input -> Prompt -> Modell -> Parser -> Nachverarbeitung`


In [ ]:
# Mini-Check: Ein einfacher RunnableLambda-Schritt
normalize = RunnableLambda(lambda x: x.strip().lower())
print(normalize.invoke("   HALLO LCEL   "))

# 3 | Pipe-Operator
---

Mit `|` verbinden Sie Schritte zu einer Kette.
Das macht Datenflüsse explizit und gut testbar.


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Erstelle 3 Stichpunkte zu folgendem Thema: {topic}"
)

basic_chain = prompt | llm | parser

result = basic_chain.invoke({"topic": "Vorteile von Unit Tests"})

mprint('### Ergebnis')
mprint('---')
mprint(result)

# 4 | Minimalbeispiel: prompt | llm
---



Ohne Parser erhalten Sie ein AIMessage-Objekt.
Mit `StrOutputParser()` arbeiten Sie direkt mit Text weiter.


In [ ]:
chain_raw = prompt | llm
raw = chain_raw.invoke({"topic": "Python Dataclasses"})

mprint('### Ergebnis')
mprint('---')
mprint(raw.content)

# 5 | Komplexe Chains
---

`RunnableParallel` erlaubt parallele Teilketten.
So lassen sich z. B. Zusammenfassung und Kritik gleichzeitig erzeugen.


In [ ]:
base_prompt = ChatPromptTemplate.from_template(
    "Analysiere den folgenden Text: {text}"
)

summary_chain = (
    base_prompt
    | llm
    | parser
    | RunnableLambda(lambda t: f"Zusammenfassung:\n{t}")
)

critic_chain = (
    ChatPromptTemplate.from_template(
        "Nenne 3 mögliche Schwächen in diesem Text:\n{text}"
    )
    | llm
    | parser
    | RunnableLambda(lambda t: f"Kritik:\n{t}")
)

combined_chain = RunnableParallel(
    summary=summary_chain,
    critique=critic_chain,
)

text = "LCEL ist praktisch, weil Chains deklarativ und modular aufgebaut werden können."
out = combined_chain.invoke({"text": text})

mprint('### Ergebnis')
mprint('---')
mprint(out["summary"])
mprint('---')
mprint(out["critique"])

In [ ]:
# LangSmith: Aktives Projekt für diesen Abschnitt
import os
print(f"📊 LangSmith-Projekt: {os.environ['LANGCHAIN_PROJECT']}")

# 6 | Chain-Debugging mit LangSmith
---

Mit LangSmith sehen Sie:
- welche Chain-Schritte aufgerufen wurden
- welche Inputs/Outputs pro Schritt entstanden sind
- wo Fehler oder unerwartete Antworten auftreten


<p><font color='black' size="5">
LangSmith konfigurieren
</font></p>

**Hinweis:** Verwenden Sie den EU-API-Endpoint `https://eu.api.smith.langchain.com`.

In [ ]:
# Konfiguration bestätigen (Vars in Setup-Cell gesetzt)
import os
print(f"Tracing aktiv: {os.getenv('LANGCHAIN_TRACING_V2')}")
print(f"Endpunkt:      {os.getenv('LANGCHAIN_ENDPOINT')}")
print(f"Projekt:       {os.getenv('LANGCHAIN_PROJECT')}")

<p><font color='black' size="5">
Tracing
</font></p>

In [ ]:
# Beispiel-Komponenten (falls noch nicht definiert)
prompt = ChatPromptTemplate.from_template("Erzähle mir kurz etwas über {topic}")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M05_Kap6_Chain_Debugging",
    "tags":     ["M05", "lcel", "chain", "debug"]
}

# 2. with_config() anwenden, dann invoke()
debug_chain = (prompt | llm | parser).with_config(**run_cfg)
dbg = debug_chain.invoke({"topic": "Wann lohnt sich Caching?"})

mprint("### Trace erstellt")
mprint('---')
mprint(dbg)

# 7 | Streaming mit LCEL

- Unterschied `invoke()` vs. `stream()`/`astream()`
- Wann Streaming UX verbessert (lange Antworten, Live-Feedback)
- Mini-Demo: gleiche Chain einmal normal, einmal gestreamt

---


In [ ]:
stream_prompt = ChatPromptTemplate.from_template(
    "Erkläre das Konzept '{topic}' in 5 kurzen Absätzen."
)
stream_chain = stream_prompt | llm | parser

mprint("### Normaler Aufruf (invoke):")
mprint('---')
full = stream_chain.invoke({"topic": "Event-getriebene Agenten"})
mprint(full)

In [ ]:
mprint("### Gestreamter Aufruf (stream):")
mprint('---')
for chunk in stream_chain.stream({"topic": "Event-getriebene Agenten"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M05-LCEL-Chains", limit=3, show_steps=True)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.


1. Bauen Sie eine LCEL-Chain mit mindestens 4 Schritten (Prompt -> LLM -> Parser -> Postprocessing).
2. Ergänzen Sie eine zweite Teilkette und kombinieren Sie beide mit `RunnableParallel`.
3. Implementieren Sie eine kleine Bewertungsfunktion (`RunnableLambda`), die die Antwortlänge oder Struktur prüft.
4. Testen Sie die Chain mit 3 unterschiedlichen Inputs und notieren Sie Auffälligkeiten.

